In [25]:
import sys
sys.path.append('../utilities/')
import pandas as pd
import numpy as np
from sklearn import svm
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
import torch
from sentence_transformers import SentenceTransformer
from joblib import dump
from openai import OpenAI
from tqdm import tqdm
from mmd import MMD
import re
from sklearn.feature_extraction.text import CountVectorizer
import tiktoken
from collections import defaultdict, Counter
import os
from dotenv import load_dotenv

# **Sentence Transformer**

In [2]:
sentence_transformer = SentenceTransformer('all-mpnet-base-v2')

# **Data Pre-processing**

In [3]:
train_df = pd.read_csv('../data/initial_datasets/reddit/reddit_train.csv')
test_df = pd.read_csv('../data/initial_datasets/reddit/reddit_test.csv')

In [4]:
train_df = train_df.sample(n=1000)

# **Tokenizer**

In [5]:
encoding = tiktoken.encoding_for_model("gpt-4")

# **LLM**

In [26]:
load_dotenv()
API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=API_KEY)

# **Get Co-Occurences**

In [9]:
text = train_df['text'].to_list()

In [10]:
tokens_list = []
for sentence in text:
    token_ids = encoding.encode(sentence)
    # Optionally, get string versions of tokens
    tokens = [encoding.decode([tid]) for tid in token_ids]
    tokens_list.append(tokens)

In [12]:
cooc = defaultdict(Counter)

for tokens in tokens_list:
    unique_tokens = set(tokens)
    for token in unique_tokens:
        for other_token in unique_tokens:
            if token != other_token:
                cooc[token][other_token] += 1

In [19]:
top_k = 3

summary_text = ""
for token, counter in cooc.items():
    top = [w for w, _ in counter.most_common(top_k)]
    summary_text += f"'{token}' often appears with: {', '.join(top)}.\n"


In [20]:
print(summary_text)

' the' often appears with: ., ,,  I.
' you' often appears with: ., ,,  the.
',' often appears with: .,  the,  I.
' glad' often appears with:  I, ., ,.
' a' often appears with: .,  the, ,.
' I' often appears with: .,  the, ,.
' could' often appears with: .,  I,  that.
'’t' often appears with: ., ,,  the.
' fantastic' often appears with:  the,  you, ,.
' don' often appears with: ., 't,  the.
' and' often appears with: .,  the,  a.
' Have' often appears with:  a, .,  the.
'Well' often appears with: ., ,,  I.
' str' often appears with:  the,  you, ,.
'.' often appears with:  the,  I,  a.
'a' often appears with:  the,  I, I.
' scare' often appears with:  the,  you, ,.
' day' often appears with: .,  the,  a.
'ights' often appears with:  the,  you, ,.
'!' often appears with: .,  I,  the.
' let' often appears with: .,  the,  a.
' help' often appears with: .,  I,  the.
' away' often appears with:  the, .,  is.
' ' often appears with: ., ,,  the.
'50' often appears with:  the,  ,  I.
' so' often

In [21]:
instruction = (
    "You are a data generator tasked with creating realistic Reddit comments. "
    "These comments should be labeled according to their sentiment: positive or negative.\n"
    "Base the style on typical Reddit comments — include informal internet language, typos, abbreviations, and emojis.\n"
    "You will be given information about token co-occurences which provides information on which words appear near each other.\n"
    "Use [NAME] as a placeholder anytime a person's name would appear.\n"
    "Generate exactly 10 realistic Reddit comments, one per line.\n"
    "Each line should follow this format: the comment in double quotes, followed by a space and then the label (1 for positive, -1 for negative).\n"
    "No extra formatting — just plain text output, one line per comment.\n"
    "Here is the format:\n"
    "\"I love pizza\" 1\n"
    "\"I hate baseball\" -1"
)
input = (
    f"Here are the token co-occurences ordered by frequency:\n{summary_text}",
    f"Now, generate the 10 new comments below:"
)

In [36]:
response = client.responses.create(
        model="gpt-4o",
        instructions=instruction,
        input=input[0]
    )
print(response.output_text)

1. "Well, I’m glad you enjoyed the game!" 1
2. "I’m so tired of this ridiculous weather." -1
3. "Fantastic job on the project, keep it up!" 1
4. "I can’t believe how bad the movie was..." -1
5. "Honestly, that was the most amazing story!" 1
6. "The customer service here is just awful." -1
7. "I hope your day is going great so far." 1
8. "Why is this game always bugging out? Ugh." -1
9. "You’re doing a fantastic job, congrats!" 1
10. "Ugh, I can’t stand the traffic here." -1


In [44]:
res = []
for i in tqdm(range(30)):
    response = client.responses.create(
        model="gpt-4o",
        instructions=instruction,
        input=input[0]
    )
    res.append(response.output_text)

  0%|          | 0/30 [00:00<?, ?it/s]

100%|██████████| 30/30 [05:42<00:00, 11.43s/it]


In [41]:
res

['"I’m so glad I found this community! Everyone here is super helpful and welcoming. 😊" 1  \n"Ugh, this really sucks. I can\'t believe people actually enjoy this nonsense." -1  \n"Wow, I just finished that book and it was absolutely fantastic! Couldn\'t put it down!" 1  \n"I thought my day couldn\'t get any worse, then I read this mess. 😒" -1  \n"I can\'t express how happy I am. Everything went so smoothly today!" 1  \n"This company’s management is so frustrating, nothing ever gets done right. 😠" -1  \n"Glad I joined this project, the team is amazing and I’ve learned a ton!" 1  \n"Honestly, this has been a complete waste of time. Can\'t recommend it at all." -1  \n"Had a great time at the event yesterday. Met some awesome people and learned lots!" 1  \n"I hate when things are so disorganized. It’s just exhausting dealing with it." -1  ',
 '"I\'m so glad you shared that experience! 😊" 1  \n"Honestly, it was a terrible movie." -1  \n"Your post made my day! Thanks for sharing!" 1  \n"Ugh,

In [45]:
labels = []
sentences = []
for i in range(30):
    for word in res[i].split("\n"):
        match = re.match(r'"(.*?)"\s*(-?\d+)', word)
        if match:
            quoted = match.group(1)      
            label = match.group(2)       
            sentences.append(quoted)
            labels.append(int(label))

In [46]:
generated_df = pd.DataFrame({
    'sentences': sentences,
    'labels': labels
})

In [47]:
generated_df

,sentences,labels
0,I’m glad that movie was so entertaining!,1
1,"Ugh, I can't stand when they mess up orders.",-1
2,"Great game last night, the players were fantas...",1
3,Why does this keep happening? So annoying.,-1
4,Their new album dropped and it's amazing!!,1
...,...,...
245,"Another day, another boring meeting 😒",-1
246,"So glad I went for a walk today, feeling refre...",1
247,That game last night was pretty disappointing 😞,-1
248,Just finished a great book and loved every page!,1


In [48]:
first_df

,sentences,labels
0,I’m so glad I found this community! Everyone h...,1
1,"Ugh, this really sucks. I can't believe people...",-1
2,"Wow, I just finished that book and it was abso...",1
3,"I thought my day couldn't get any worse, then ...",-1
4,I can't express how happy I am. Everything wen...,1
...,...,...
855,"Well, that’s just a nightmare scenario, sorry.",-1
856,Happy to see improvement in their game.,1
857,I don’t know how to deal with all this stress.,-1
858,"It’s great seeing you here, welcome back!",1


In [50]:
combined = pd.concat([first_df, generated_df])

In [52]:
combined.to_csv('../data/generated/reddit/token_co_occurences/token_co_occurences.csv', index=False)